# Sector Rotation as a Markov Chain

## 0. Setup

In [1]:
import wrds
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, chi2_contingency, chi2 as chi2dist

In [2]:
SECTORS = ["Energy", "Materials", "Industrials", "ConsDiscretionary",
           "ConsStaples", "InfoTech", "HealthCare", "Financials",
           "CommServices", "Utilities", "RealEstate"]
GVKEYX_MAP = {
    "128798": "Energy", "128859": "Materials", "128898": "Industrials",
    "128940": "ConsDiscretionary", "129001": "ConsStaples",
    "129059": "InfoTech", "129021": "HealthCare", "129039": "Financials",
    "129081": "CommServices", "129218": "Utilities", "027228": "RealEstate",
}
START = "2016-09-19"
ALPHA = 1.0
RNG_SEED = 42
M = len(SECTORS)
pos = {s: i for i, s in enumerate(SECTORS)}
rng = np.random.default_rng(RNG_SEED)

In [3]:
def stationary_distribution(P_mat, states):
    vals, vecs = np.linalg.eig(P_mat.T)
    i = np.argmin(np.abs(vals - 1.0))
    pi = np.real(vecs[:, i]); pi = pi / pi.sum()
    return pd.Series(pi, index=states), vals[i]

def lambda2_from_sequence(seq_list, states, alpha=ALPHA):
    p = {s: i for i, s in enumerate(states)}; mm = len(states)
    Nm = np.zeros((mm, mm))
    for a, b in zip(seq_list[:-1], seq_list[1:]):
        Nm[p[a], p[b]] += 1
    Pm = (Nm + alpha) / (Nm + alpha).sum(axis=1, keepdims=True)
    ev = np.linalg.eigvals(Pm); ev = ev[np.argsort(-np.abs(ev))]
    return np.abs(ev[1])

def eig_summary(Pm, label):
    vals, vecs = np.linalg.eig(Pm)
    order = np.argsort(-np.abs(vals))
    vals = vals[order]
    vecs = vecs[:, order]
    mod = np.abs(vals)
    print(f"--- {label} ---")
    tab = pd.DataFrame({
        "Re": np.round(vals.real, 5),
        "Im": np.round(vals.imag, 5),
        "modulus": np.round(mod, 5),
        "is_complex": np.abs(vals.imag) > 1e-9,
    })
    print(tab.to_string())
    print("sum of eigenvalues (=trace):", round(vals.sum().real, 5),
          "| trace(P):", round(np.trace(Pm), 5))
    print("largest eigenvalue:", np.round(vals[0], 8))
    print("number of complex eigenvalues:", int((np.abs(vals.imag) > 1e-9).sum()))
    return vals, vecs, mod

## 1. Data Acquisition

In [4]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [5]:
gvkeyx_str = ", ".join(f"'{g}'" for g in GVKEYX_MAP)

query = f"""
    SELECT datadate, gvkeyx, prccddiv, prccddivn, prccd, dvpsxd
    FROM comp.idx_daily
    WHERE gvkeyx IN ({gvkeyx_str})
    ORDER BY gvkeyx, datadate
"""
raw = db.raw_sql(query, date_cols=["datadate"])

print("rows pulled:", raw.shape[0])
print("sectors:", raw["gvkeyx"].nunique(), "/ 11")
print("date range:", raw["datadate"].min().date(), "->", raw["datadate"].max().date())

rows pulled: 95050
sectors: 11 / 11
date range: 1989-09-11 -> 2026-06-17


## 2. Data Preparation & Validation

Common sample starts 2016-09-19: Real Estate split from Financials into a standalone GICS sector then. Other 10 sectors run back to 1989 but are truncated to keep one consistent 11-sector panel.

In [20]:
df = raw.copy()
df["sector"] = df["gvkeyx"].map(GVKEYX_MAP)
df["datadate"] = pd.to_datetime(df["datadate"])
df["prccddiv"] = pd.to_numeric(df["prccddiv"], errors="coerce")
df["prccd"] = pd.to_numeric(df["prccd"], errors="coerce")
df = df.sort_values(["gvkeyx", "datadate"])
df["ret"] = df.groupby("gvkeyx")["prccddiv"].apply(lambda s: np.log(s).diff()).values

print("duplicate (sector, date) rows:",
      int(df.duplicated(subset=["gvkeyx", "datadate"]).sum()))
print("non-positive / null TR levels:",
      int(((df["prccddiv"].isna()) | (df["prccddiv"] <= 0)).sum()))
print("daily log-return outliers |ret|>0.25:",
      int((df["ret"].abs() > 0.25).sum()))
print("exact-zero returns (stale):", int((df["ret"] == 0).sum()))
print("weekend-dated rows:", int((df["datadate"].dt.dayofweek >= 5).sum()))

tr_pr_ok = True
for g, sub in df.groupby("gvkeyx"):
    tr = sub["prccddiv"].iloc[-1] / sub["prccddiv"].iloc[0] - 1
    pr = sub["prccd"].iloc[-1] / sub["prccd"].iloc[0] - 1
    if tr < pr:
        tr_pr_ok = False
print("cumulative TR >= price return for all sectors:", tr_pr_ok)

duplicate (sector, date) rows: 0
non-positive / null TR levels: 0
daily log-return outliers |ret|>0.25: 0
exact-zero returns (stale): 0
weekend-dated rows: 0
cumulative TR >= price return for all sectors: True


In [21]:
panel = (df[df["datadate"] >= START]
         .pivot_table(index="datadate", columns="sector",
                      values="prccddiv", aggfunc="first")
         .sort_index()[SECTORS]
         .astype("float64"))

rets = np.log(panel).diff().dropna(how="any")

print("panel shape:", panel.shape, "| return panel:", rets.shape)
print("date range:", panel.index.min().date(), "->", panel.index.max().date())
print("any NaN (level / return):",
      int(panel.isna().sum().sum()), "/", int(rets.isna().sum().sum()))
print("strictly increasing dates:", bool((panel.index[1:] > panel.index[:-1]).all()))
print("all-11-present rows:", int(panel.notna().all(axis=1).sum()), "/", panel.shape[0])

panel shape: (2450, 11) | return panel: (2449, 11)
date range: 2016-09-19 -> 2026-06-17
any NaN (level / return): 0 / 0
strictly increasing dates: True
all-11-present rows: 2450 / 2450


## 3. State Construction & Transition Matrix

In [8]:
weekly_ret = rets.resample("W-FRI").sum()
weekly_ret = weekly_ret[rets.resample("W-FRI").size() > 0]

state = weekly_ret.idxmin(axis=1)
state.name = "laggard"
seq = state.tolist()

print("weeks (states):", len(state))
print("trading-days-per-week:",
      rets.resample("W-FRI").size().value_counts().sort_index().to_dict())
print("ties (>1 sector at min):",
      int((weekly_ret.eq(weekly_ret.min(axis=1), axis=0)).sum(axis=1).gt(1).sum()))
print("\nlaggard frequency by sector:")
print(state.value_counts().reindex(SECTORS).to_string())

weeks (states): 509
trading-days-per-week: {3: 1, 4: 94, 5: 414}
ties (>1 sector at min): 0

laggard frequency by sector:
laggard
Energy               127
Materials             27
Industrials           13
ConsDiscretionary     40
ConsStaples           34
InfoTech              43
HealthCare            34
Financials            34
CommServices          53
Utilities             61
RealEstate            43


In [9]:
N = np.zeros((M, M), dtype=int)
for a, b in zip(seq[:-1], seq[1:]):
    N[pos[a], pos[b]] += 1

row_tot = N.sum(axis=1)
P = np.divide(N, row_tot[:, None], where=row_tot[:, None] > 0,
              out=np.zeros((M, M)))

N_df = pd.DataFrame(N, index=SECTORS, columns=SECTORS)
P_df = pd.DataFrame(P, index=SECTORS, columns=SECTORS)

print("total transitions:", N.sum(), "(expected", len(seq) - 1, ")")
print("zero-count cells:", int((N == 0).sum()), "/", M * M)
print("zero origin-rows:", [SECTORS[i] for i in range(M) if row_tot[i] == 0])
print("row sums all 1:", np.allclose(P.sum(axis=1), 1.0))
print("\nsample size per origin state (n_i):")
print(pd.Series(row_tot, index=SECTORS).to_string())

total transitions: 508 (expected 508 )
zero-count cells: 7 / 121
zero origin-rows: []
row sums all 1: True

sample size per origin state (n_i):
Energy               126
Materials             27
Industrials           13
ConsDiscretionary     40
ConsStaples           34
InfoTech              43
HealthCare            34
Financials            34
CommServices          53
Utilities             61
RealEstate            43


In [10]:
free_params = M * (M - 1)
print("free parameters in P:", free_params)
print("transitions per parameter — daily: {:.2f} | weekly: {:.2f}".format(
    (len(rets) - 1) / free_params, (len(state) - 1) / free_params))

N_smooth = N + ALPHA
P_smooth = N_smooth / N_smooth.sum(axis=1, keepdims=True)
P_smooth_df = pd.DataFrame(P_smooth, index=SECTORS, columns=SECTORS)

print("\nzero cells: before =", int((N == 0).sum()),
      "| after smoothing =", int((P_smooth == 0).sum()))
print("row sums after smoothing all 1:", np.allclose(P_smooth.sum(axis=1), 1.0))
print("max per-cell change vs raw:", round(np.nanmax(np.abs(P_smooth - P)), 4))
print("\nrows most shifted by smoothing (L1, smallest-sample states move most):")
l1 = pd.Series(np.abs(P_smooth - P).sum(axis=1), index=SECTORS)
print(l1.sort_values(ascending=False).round(3).to_string())

free parameters in P: 110
transitions per parameter — daily: 22.25 | weekly: 4.62

zero cells: before = 7 | after smoothing = 0
row sums after smoothing all 1: True
max per-cell change vs raw: 0.0702

rows most shifted by smoothing (L1, smallest-sample states move most):
Industrials          0.288
Materials            0.187
Financials           0.167
ConsStaples          0.124
HealthCare           0.123
InfoTech             0.117
ConsDiscretionary    0.113
RealEstate           0.100
CommServices         0.094
Utilities            0.077
Energy               0.038


Weekly transitions per parameter = 4.6 (110 params, 508 transitions). Sparse. 7 zero cells removed by Laplace α=1; smoothing shifts low-sample rows (Industrials, Materials) most.

In [11]:
print("[1] state vs recomputed idxmin:",
      bool((state == weekly_ret.idxmin(axis=1)).all()))

print("[2] ties in idxmin:",
      int((weekly_ret.eq(weekly_ret.min(axis=1), axis=0)).sum(axis=1).gt(1).sum()))

N2 = np.zeros((M, M), dtype=int)
for a, b in zip(seq[:-1], seq[1:]):
    N2[pos[a], pos[b]] += 1
print("[3] N independent rebuild matches:", bool((N == N2).all()))

P_check = (N + ALPHA) / (N + ALPHA).sum(axis=1, keepdims=True)
print("[4] P_smooth traceable to (N+alpha):", np.allclose(P_smooth, P_check))

wk = rets.resample("W-FRI").sum()
wk = wk[rets.resample("W-FRI").size() > 0]
recon = np.exp(wk.sum(axis=0)) - 1
direct = panel.iloc[-1] / panel.iloc[0] - 1
print("[5] log-return aggregation exact:",
      round(float((recon - direct).abs().max()), 10))

print("[6] column order consistent:",
      list(panel.columns) == SECTORS == list(rets.columns) == list(weekly_ret.columns))

print("[7] resample no row leak:",
      len(rets) == int(rets.resample("W-FRI").size().sum()))

[1] state vs recomputed idxmin: True
[2] ties in idxmin: 0
[3] N independent rebuild matches: True
[4] P_smooth traceable to (N+alpha): True
[5] log-return aggregation exact: 0.0
[6] column order consistent: True
[7] resample no row leak: True


## 4. Markov Dynamics

In [12]:
pi_raw, ev_raw = stationary_distribution(P, SECTORS)
pi_smooth, ev_smooth = stationary_distribution(P_smooth, SECTORS)

comp = pd.DataFrame({"raw": pi_raw, "smooth": pi_smooth}).round(4)
comp["empirical_freq"] = (state.value_counts().reindex(SECTORS) / len(state)).round(4)

print("eigenvalue used (should be ~1):  raw", np.round(ev_raw, 6),
      "| smooth", np.round(ev_smooth, 6))
print("\nstationary pi (raw vs smoothed vs empirical):")
print(comp.to_string())

print("\nsanity:")
print("  pi sums to 1 (raw/smooth):", round(pi_raw.sum(), 6), "/", round(pi_smooth.sum(), 6))
print("  any negative (raw/smooth):", bool((pi_raw < 0).any()), "/", bool((pi_smooth < 0).any()))
print("  residual |pi P - pi| (smooth):",
      round(np.max(np.abs(pi_smooth.values @ P_smooth - pi_smooth.values)), 10))

v = np.ones(M) / M
for _ in range(100000):
    v_new = v @ P_smooth
    if np.max(np.abs(v_new - v)) < 1e-14:
        break
    v = v_new
print("  [E] power-iteration vs eigvec pi (max abs diff):",
      round(float(np.max(np.abs(v - pi_smooth.values))), 12))

eigenvalue used (should be ~1):  raw (1+0j) | smooth (1+0j)

stationary pi (raw vs smoothed vs empirical):
                      raw  smooth  empirical_freq
Energy             0.2480  0.2178          0.2495
Materials          0.0531  0.0604          0.0530
Industrials        0.0256  0.0382          0.0255
ConsDiscretionary  0.0787  0.0811          0.0786
ConsStaples        0.0669  0.0715          0.0668
InfoTech           0.0846  0.0859          0.0845
HealthCare         0.0669  0.0715          0.0668
Financials         0.0669  0.0715          0.0668
CommServices       0.1043  0.1017          0.1041
Utilities          0.1201  0.1145          0.1198
RealEstate         0.0846  0.0859          0.0845

sanity:
  pi sums to 1 (raw/smooth): 1.0 / 1.0
  any negative (raw/smooth): False / False
  residual |pi P - pi| (smooth): 0.0
  [E] power-iteration vs eigvec pi (max abs diff): 0.0


Stationary π matches empirical frequencies → self-consistent. Energy dominates (~25%), reflecting 2016–20 oil weakness.

In [13]:
recurrence = 1.0 / pi_smooth

def hitting_times_to(target_idx, Pm):
    others = [i for i in range(M) if i != target_idx]
    A = np.eye(len(others)) - Pm[np.ix_(others, others)]
    h_other = np.linalg.solve(A, np.ones(len(others)))
    h = np.zeros(M)
    for p_i, i in enumerate(others):
        h[i] = h_other[p_i]
    return h

H = np.column_stack([hitting_times_to(j, P_smooth) for j in range(M)])
H_df = pd.DataFrame(H, index=SECTORS, columns=SECTORS)

print("mean recurrence time (1/pi), weeks:")
print(recurrence.round(2).sort_values().to_string())
print("\nhitting time H[i,j] = mean weeks from i to first reach j:")
print(H_df.round(1).to_string())

ret_via_hit = np.array([1 + P_smooth[j, :] @ H[:, j] for j in range(M)])
print("\nsanity:")
print("  diagonal of H all zero:", np.allclose(np.diag(H), 0))
print("  any negative hitting time:", bool((H < 0).any()))
print("  Kac identity max |1/pi - (1 + sum_k P_jk h_kj)|:",
      round(float(np.max(np.abs(recurrence.values - ret_via_hit))), 10))

mean recurrence time (1/pi), weeks:
Energy                4.59
Utilities             8.74
CommServices          9.83
InfoTech             11.65
RealEstate           11.65
ConsDiscretionary    12.33
ConsStaples          13.98
HealthCare           13.98
Financials           13.98
Materials            16.55
Industrials          26.21

hitting time H[i,j] = mean weeks from i to first reach j:
                   Energy  Materials  Industrials  ConsDiscretionary  ConsStaples  InfoTech  HealthCare  Financials  CommServices  Utilities  RealEstate
Energy                0.0       16.4         26.8               11.8         13.4      11.7        14.9        15.1          10.2        8.0        12.3
Materials             4.4        0.0         26.7               12.1         14.1      11.5        14.5        14.1          10.8        8.4        12.0
Industrials           5.0       16.7          0.0               12.1         14.4      11.3        13.8        14.3           9.8        8.7        1

In [14]:
vals_raw, vecs_raw, mod_raw = eig_summary(P, "RAW P-hat")
print()
vals_s, vecs_s, mod_s = eig_summary(P_smooth, "SMOOTHED P-hat")

print("\ncross-checks (smoothed):")
print("  Perron (largest modulus):", round(mod_s[0], 8))
print("  all moduli <= 1:", bool((mod_s <= 1 + 1e-9).all()))
print("  |lambda_2|:", round(mod_s[1], 5),
      "| spectral gap g = 1 - |lambda_2|:", round(1 - mod_s[1], 5))
print("  relaxation time ~ 1/g (weeks):", round(1 / (1 - mod_s[1]), 2))
print("  max imag part (cycle indicator):",
      round(float(np.max(np.abs(vals_s.imag))), 5))
print("  complex eigenvalues come in conjugate pairs:",
      np.allclose(sorted(vals_s.imag), sorted(-vals_s.imag)))

--- RAW P-hat ---
         Re       Im  modulus  is_complex
0   1.00000  0.00000  1.00000       False
1   0.11761  0.01288  0.11831        True
2   0.11761 -0.01288  0.11831        True
3   0.01250  0.10250  0.10326        True
4   0.01250 -0.10250  0.10326        True
5  -0.07579  0.05295  0.09245        True
6  -0.07579 -0.05295  0.09245        True
7  -0.09158  0.00000  0.09158       False
8   0.01385  0.03756  0.04003        True
9   0.01385 -0.03756  0.04003        True
10 -0.00327  0.00000  0.00327       False
sum of eigenvalues (=trace): 1.04152 | trace(P): 1.04152
largest eigenvalue: (1+0j)
number of complex eigenvalues: 8

--- SMOOTHED P-hat ---
         Re       Im  modulus  is_complex
0   1.00000  0.00000  1.00000       False
1   0.09572  0.01311  0.09661        True
2   0.09572 -0.01311  0.09661        True
3  -0.07696  0.00000  0.07696       False
4   0.00446  0.07666  0.07679        True
5   0.00446 -0.07666  0.07679        True
6  -0.04636  0.02174  0.05121        True
7

|λ₂|=0.097, spectral gap 0.90, relaxation ~1 week. Max imag part 0.08 → no cyclical rotation. Chain is near-memoryless.

## 5. Signal vs Noise

Laggard |λ₂| inside null (p=0.88) → no memory. Positive control confirms detection works. Leader flags under simple shuffle (p=0.022) but dies under daily re-test (p=0.83) and block bootstrap (p=0.12): artifact. Raw-return lead-lag negligible after market-demeaning (daily mean|c|=0.024, placebo flat). No usable signal.

In [15]:
actual_l2 = lambda2_from_sequence(seq, SECTORS)
print("actual |lambda_2| (smoothed):", round(actual_l2, 5))

print("\n[1] PERMUTATION TEST (shuffle destroys time-structure)")
B = 2000
null_l2 = np.empty(B)
arr = np.array(seq, dtype=object)
for b in range(B):
    null_l2[b] = lambda2_from_sequence(list(rng.permutation(arr)), SECTORS)
p_val = (np.sum(null_l2 >= actual_l2) + 1) / (B + 1)
print("  null mean / 97.5%:", round(null_l2.mean(), 5), "/", round(np.percentile(null_l2, 97.5), 5))
print(f"  actual = {actual_l2:.5f} | p = {p_val:.4f}")
print("  verdict:", "NOISE (inside band)" if actual_l2 <= np.percentile(null_l2, 97.5) else "SIGNAL")

print("\n[2] POSITIVE CONTROL (inject a 4-sector cycle)")
cycle = ["Energy", "Utilities", "InfoTech", "Financials"]
p_stay = 0.15
syn = [cycle[0]]
for _ in range(len(seq) - 1):
    cur = syn[-1]
    if cur in cycle and rng.random() > p_stay:
        nxt = cycle[(cycle.index(cur) + 1) % len(cycle)]
    else:
        nxt = rng.choice(SECTORS)
    syn.append(nxt)

Nsyn = np.zeros((M, M))
for a, b in zip(syn[:-1], syn[1:]):
    Nsyn[pos[a], pos[b]] += 1
Psyn = (Nsyn + ALPHA) / (Nsyn + ALPHA).sum(axis=1, keepdims=True)
ev_syn = np.linalg.eigvals(Psyn)
ev_syn = ev_syn[np.argsort(-np.abs(ev_syn))]

print("  injected cycle:", " -> ".join(cycle), "-> (repeat)")
print("  synthetic |lambda_2|:", round(abs(ev_syn[1]), 5))
print("  synthetic max imag part:", round(float(np.max(np.abs(ev_syn.imag))), 5),
      "(large => cycle detected)")
print("  real-data max imag part:", round(float(np.max(np.abs(vals_s.imag))), 5),
      "(small => no cycle)")

print("\n[3] STATE-DEFINITION ROBUSTNESS: laggard vs leader")
leader_state = weekly_ret.idxmax(axis=1)

def full_diag(seq_series, label, nb=2000):
    sl = seq_series.tolist()
    Nm = np.zeros((M, M))
    for a, b in zip(sl[:-1], sl[1:]):
        Nm[pos[a], pos[b]] += 1
    Pm = (Nm + ALPHA) / (Nm + ALPHA).sum(axis=1, keepdims=True)
    ev = np.linalg.eigvals(Pm); ev = ev[np.argsort(-np.abs(ev))]
    l2 = abs(ev[1])
    arr = np.array(sl, dtype=object)
    nullv = np.empty(nb)
    for b in range(nb):
        nullv[b] = lambda2_from_sequence(list(rng.permutation(arr)), SECTORS)
    pv = (np.sum(nullv >= l2) + 1) / (nb + 1)
    c975 = np.percentile(nullv, 97.5)
    print(f"  {label:28s} |l2|={l2:.5f}  null97.5={c975:.5f}  "
          f"maxIm={np.max(np.abs(ev.imag)):.4f}  p={pv:.4f}  "
          f"{'SIGNAL' if l2 > c975 else 'NOISE'}")

full_diag(state, "LAGGARD (argmin) [main]")
full_diag(leader_state, "LEADER (argmax)")

print("\n[4] CHI-SQUARE TEST OF FIRST-ORDER DEPENDENCE")
print("  H0: next state independent of current state")
for lbl, sq in [("LAGGARD", seq), ("LEADER", leader_state.tolist())]:
    Nm = np.zeros((M, M))
    for a, b in zip(sq[:-1], sq[1:]):
        Nm[pos[a], pos[b]] += 1
    Nuse = Nm[Nm.sum(axis=1) > 0][:, Nm.sum(axis=0) > 0]
    chi2_stat, p, dof, exp = chi2_contingency(Nuse)
    print(f"  {lbl:8s} chi2={chi2_stat:.2f}  dof={dof}  p={p:.4f}  "
          f"cells_exp<5={ (exp<5).mean():.0%}  "
          f"{'DEPENDENT' if p < 0.05 else 'INDEPENDENT'}")
    

print("\n[5] DAILY-FREQUENCY RE-TEST (5x sample, lower noise floor)")
daily_lag = rets.idxmin(axis=1)
daily_led = rets.idxmax(axis=1)

def l2_perm(seq_list, nb=2000):
    Nm = np.zeros((M, M))
    for a, b in zip(seq_list[:-1], seq_list[1:]):
        Nm[pos[a], pos[b]] += 1
    Pm = (Nm + ALPHA) / (Nm + ALPHA).sum(axis=1, keepdims=True)
    ev = np.linalg.eigvals(Pm); ev = ev[np.argsort(-np.abs(ev))]
    l2 = abs(ev[1])
    arr = np.array(seq_list, dtype=object)
    nullv = np.array([lambda2_from_sequence(list(rng.permutation(arr)), SECTORS)
                      for _ in range(nb)])
    pv = (np.sum(nullv >= l2) + 1) / (nb + 1)
    return l2, np.percentile(nullv, 97.5), pv, ev

for lbl, sq in [("DAILY laggard", daily_lag.tolist()), ("DAILY leader", daily_led.tolist())]:
    l2, c975, pv, ev = l2_perm(sq)
    print(f"  {lbl:14s} |l2|={l2:.5f}  null97.5={c975:.5f}  "
          f"maxIm={np.max(np.abs(ev.imag)):.4f}  p={pv:.4f}  "
          f"{'SIGNAL' if l2 > c975 else 'NOISE'}")

print("\n[6] WEEKLY LEADER lambda_2: real or complex?")
Nl = np.zeros((M, M))
for a, b in zip(leader_state.tolist()[:-1], leader_state.tolist()[1:]):
    Nl[pos[a], pos[b]] += 1
Pl = (Nl + ALPHA) / (Nl + ALPHA).sum(axis=1, keepdims=True)
ev_l = np.linalg.eigvals(Pl); ev_l = ev_l[np.argsort(-np.abs(ev_l))]
print(f"  lambda_2 = Re {ev_l[1].real:+.5f}  Im {ev_l[1].imag:+.5f}  "
      f"({'complex/cycle' if abs(ev_l[1].imag) > 1e-6 else 'real/persistence'})")
print(f"  mean diagonal P[i,i]: {np.mean(np.diag(Pl)):.4f}  | uniform 1/M = {1/M:.4f}")

print("\n[7] BLOCK BOOTSTRAP null (preserves short-run autocorrelation)")
def block_l2(seq_list, blocklen=4, nb=2000):
    Nm = np.zeros((M, M))
    for a, b in zip(seq_list[:-1], seq_list[1:]):
        Nm[pos[a], pos[b]] += 1
    Pm = (Nm + ALPHA) / (Nm + ALPHA).sum(axis=1, keepdims=True)
    ev = np.linalg.eigvals(Pm); l2 = abs(ev[np.argsort(-np.abs(ev))][1])
    arr = np.array(seq_list, dtype=object); nn = len(arr)
    nblk = int(np.ceil(nn / blocklen))
    blocks = [arr[i*blocklen:(i+1)*blocklen] for i in range(nblk)]
    nullv = np.empty(nb)
    for b in range(nb):
        pl = list(np.concatenate([blocks[i] for i in rng.permutation(len(blocks))]))[:nn]
        Nb = np.zeros((M, M))
        for a, c in zip(pl[:-1], pl[1:]):
            Nb[pos[a], pos[c]] += 1
        Pb = (Nb + ALPHA) / (Nb + ALPHA).sum(axis=1, keepdims=True)
        evb = np.linalg.eigvals(Pb)
        nullv[b] = abs(evb[np.argsort(-np.abs(evb))][1])
    return l2, np.percentile(nullv, 97.5), (np.sum(nullv >= l2) + 1) / (nb + 1)

for lbl, sq in [("weekly laggard", seq), ("weekly leader", leader_state.tolist())]:
    l2, c975, pv = block_l2(sq)
    print(f"  {lbl:16s} |l2|={l2:.5f}  block-null97.5={c975:.5f}  p={pv:.4f}  "
          f"{'SIGNAL' if l2 > c975 else 'NOISE'}")

actual |lambda_2| (smoothed): 0.09661

[1] PERMUTATION TEST (shuffle destroys time-structure)
  null mean / 97.5%: 0.1165 / 0.16086
  actual = 0.09661 | p = 0.8766
  verdict: NOISE (inside band)

[2] POSITIVE CONTROL (inject a 4-sector cycle)
  injected cycle: Energy -> Utilities -> InfoTech -> Financials -> (repeat)
  synthetic |lambda_2|: 0.76397
  synthetic max imag part: 0.74785 (large => cycle detected)
  real-data max imag part: 0.07666 (small => no cycle)

[3] STATE-DEFINITION ROBUSTNESS: laggard vs leader
  LAGGARD (argmin) [main]      |l2|=0.09661  null97.5=0.15865  maxIm=0.0767  p=0.8756  NOISE
  LEADER (argmax)              |l2|=0.16044  null97.5=0.15931  maxIm=0.0754  p=0.0225  SIGNAL

[4] CHI-SQUARE TEST OF FIRST-ORDER DEPENDENCE
  H0: next state independent of current state
  LAGGARD  chi2=80.29  dof=100  p=0.9265  cells_exp<5=78%  INDEPENDENT
  LEADER   chi2=85.70  dof=100  p=0.8452  cells_exp<5=72%  INDEPENDENT

[5] DAILY-FREQUENCY RE-TEST (5x sample, lower noise floor)

In [16]:
print("[V1] OWN AUTOCORRELATION (is mean-reversion the driver?)")
for lbl, R in [("DAILY", rets), ("WEEKLY", weekly_ret)]:
    Rf = R.astype("float64")
    ac = np.array([np.corrcoef(Rf[c].values[:-1], Rf[c].values[1:])[0, 1] for c in Rf.columns])
    print(f"  {lbl}: lag-1 autocorr mean={ac.mean():+.4f}  "
          f"min={ac.min():+.4f}  max={ac.max():+.4f}  all_neg={bool((ac<0).all())}")

print("\n[V2] MARKET-DEMEANED lead-lag (remove common factor)")
def leadlag_demeaned(R, label, nb=2000):
    Rf = R.astype("float64")
    D = Rf.sub(Rf.mean(axis=1), axis=0)
    X = D.iloc[:-1].values; Y = D.iloc[1:].values
    T = X.shape[0]
    Xc = X - X.mean(0); Yc = Y - Y.mean(0)
    C = (Xc.T @ Yc) / (np.sqrt((Xc**2).sum(0))[:, None] * np.sqrt((Yc**2).sum(0))[None, :])
    fro = np.sqrt((C**2).sum())
    nullf = np.empty(nb)
    for b in range(nb):
        Yp = np.roll(Y, rng.integers(1, T), axis=0)
        Ypc = Yp - Yp.mean(0)
        Cb = (Xc.T @ Ypc) / (np.sqrt((Xc**2).sum(0))[:, None] * np.sqrt((Ypc**2).sum(0))[None, :])
        nullf[b] = np.sqrt((Cb**2).sum())
    p = (np.sum(nullf >= fro) + 1) / (nb + 1)
    off = C[~np.eye(M, dtype=bool)]
    print(f"  {label:7s} T={T}  off-diag mean|c|={np.abs(off).mean():.4f}  "
          f"frac_neg={(off<0).mean():.2f}  ||C||_F={fro:.4f}  "
          f"null97.5={np.percentile(nullf,97.5):.4f}  p={p:.4f}  "
          f"{'STRUCTURE' if p<0.05 else 'NONE'}")

leadlag_demeaned(rets, "DAILY")
leadlag_demeaned(weekly_ret, "WEEKLY")

print("\n[V3] PLACEBO LAG (should decay to ~0 if real 1-step structure)")
Df = rets.astype("float64").sub(rets.astype("float64").mean(axis=1), axis=0).values
for lag in [1, 5, 20, 60]:
    X = Df[:-lag]; Y = Df[lag:]
    Xc = X - X.mean(0); Yc = Y - Y.mean(0)
    C = (Xc.T @ Yc) / (np.sqrt((Xc**2).sum(0))[:, None] * np.sqrt((Yc**2).sum(0))[None, :])
    print(f"  lag={lag:>2}: ||C||_F={np.sqrt((C**2).sum()):.4f}")

[V1] OWN AUTOCORRELATION (is mean-reversion the driver?)
  DAILY: lag-1 autocorr mean=-0.0853  min=-0.1386  max=-0.0392  all_neg=True
  WEEKLY: lag-1 autocorr mean=-0.0706  min=-0.1465  max=+0.0790  all_neg=False

[V2] MARKET-DEMEANED lead-lag (remove common factor)
  DAILY   T=2448  off-diag mean|c|=0.0241  frac_neg=0.51  ||C||_F=0.3221  null97.5=0.2774  p=0.0035  STRUCTURE
  WEEKLY  T=508  off-diag mean|c|=0.0376  frac_neg=0.45  ||C||_F=0.5608  null97.5=0.5902  p=0.0665  NONE

[V3] PLACEBO LAG (should decay to ~0 if real 1-step structure)
  lag= 1: ||C||_F=0.3221
  lag= 5: ||C||_F=0.1961
  lag=20: ||C||_F=0.2371
  lag=60: ||C||_F=0.2119


## 6. Robustness

All non-Perron eigenvalues inside circular-law disk (r=0.147) → spectrum is noise, independent of the permutation tests. Chain irreducible + aperiodic → ergodic. Markov order: order-0 (i.i.d.) preferred by LR, AIC, BIC; order-2 not estimable (4.4 obs/pair). Time-homogeneity split flags p=0.039 but 74% cells <5 and no 2008 in sample, so not reliable.

In [17]:
print("[1] CIRCULAR-LAW DISK BENCHMARK")
ev_s = np.linalg.eigvals(P_smooth)
ev_s = ev_s[np.argsort(-np.abs(ev_s))]
nbar = N.sum(axis=1).mean()
R_disk = 1.0 / np.sqrt(nbar)
print(f"  avg transitions per origin row: {nbar:.1f}")
print(f"  noise disk radius ~ 1/sqrt(n_bar): {R_disk:.4f}")
print(f"  |lambda_2| observed: {abs(ev_s[1]):.4f}")
print(f"  verdict: {'INSIDE -> noise' if abs(ev_s[1]) <= R_disk else 'OUTSIDE -> signal'}")
print("  non-Perron moduli:",
      ", ".join(f"{abs(v):.3f}" for v in ev_s[1:]))
print("  all non-Perron inside disk:",
      bool(all(abs(v) <= R_disk for v in ev_s[1:])))

print("\n[2] IRREDUCIBILITY & APERIODICITY")
def is_irreducible(Pmat, tol=1e-12):
    reach = (Pmat > tol).astype(int)
    R = reach.copy(); Mx = reach.copy()
    for _ in range(M):
        Mx = (Mx @ reach > 0).astype(int)
        R = ((R + Mx) > 0).astype(int)
    return bool((R == 1).all())

print("  raw P irreducible   :", is_irreducible(P))
print("  smoothed P irreducible:", is_irreducible(P_smooth))
print("  self-loops P[i,i]>0 (raw/smooth):",
      int((np.diag(P) > 0).sum()), "/", int((np.diag(P_smooth) > 0).sum()), "of", M)

from math import gcd
def period(Pmat, tol=1e-12):
    Pk = np.eye(M); g = 0
    for k in range(1, 3 * M + 1):
        Pk = Pk @ Pmat
        if Pk[0, 0] > tol:
            g = gcd(g, k)
    return g
print("  period of chain (smoothed):", period(P_smooth),
      "=> aperiodic" if period(P_smooth) == 1 else "=> periodic")
print("  => irreducible + aperiodic = ergodic (unique pi, guaranteed convergence)")

print("\n[3] MARKOV ORDER: order-0 (i.i.d.) vs order-1")
N1 = np.zeros((M, M))
for a, b in zip(seq[:-1], seq[1:]):
    N1[pos[a], pos[b]] += 1
col = N1.sum(axis=0); tot = N1.sum()

LR = sum(2 * N1[i, j] * np.log((N1[i, j] / N1[i].sum()) / (col[j] / tot))
         for i in range(M) for j in range(M) if N1[i, j] > 0)
df_lr = (M - 1) * (M - 1)
p_lr = 1 - chi2dist.cdf(LR, df_lr)

ll1 = sum(N1[i, j] * np.log(N1[i, j] / N1[i].sum())
          for i in range(M) for j in range(M) if N1[i, j] > 0)
ll0 = sum(N1[i, j] * np.log(col[j] / tot)
          for i in range(M) for j in range(M) if N1[i, j] > 0)
k0, k1 = M - 1, M * (M - 1)
print(f"  LR={LR:.2f}  df={df_lr}  p={p_lr:.4f}  "
      f"=> {'order-1 (memory)' if p_lr < 0.05 else 'order-0 sufficient (i.i.d.)'}")
print(f"  AIC order0={-2*ll0+2*k0:.1f}  order1={-2*ll1+2*k1:.1f}  "
      f"=> {'order1' if (-2*ll1+2*k1)<(-2*ll0+2*k0) else 'order0'}")
print(f"  BIC order0={-2*ll0+k0*np.log(tot):.1f}  order1={-2*ll1+k1*np.log(tot):.1f}  "
      f"=> {'order1' if (-2*ll1+k1*np.log(tot))<(-2*ll0+k0*np.log(tot)) else 'order0'}")

print("\n[4] ORDER-2 FEASIBILITY (sparsity wall)")
pairs = list(zip(seq[:-2], seq[1:-1], seq[2:]))
seen = {}
for a, b, c in pairs:
    seen.setdefault((a, b), 0)
    seen[(a, b)] += 1
print(f"  order-2 params: {M*M} pairs x {M} = {M*M*M}")
print(f"  transitions available: {len(pairs)} | distinct pairs seen: {len(seen)}/{M*M}")
print(f"  avg obs per pair: {len(pairs)/len(seen):.2f}  => order-2 not estimable")

print("\n[5] TIME-HOMOGENEITY (first-half vs second-half split LR)")
half = len(seq) // 2
def counts(sub):
    Nx = np.zeros((M, M))
    for a, b in zip(sub[:-1], sub[1:]):
        Nx[pos[a], pos[b]] += 1
    return Nx
Na, Nb = counts(seq[:half]), counts(seq[half:])
Npool = Na + Nb
def ll(Nx):
    r = Nx.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        Pp = np.where(r > 0, Nx / r, 0)
        return np.sum(np.where(Nx > 0, Nx * np.log(np.where(Pp > 0, Pp, 1)), 0))
LR_h = 2 * (ll(Na) + ll(Nb) - ll(Npool))
df_h = int((Npool > 0).sum()) - M
p_h = 1 - chi2dist.cdf(LR_h, max(df_h, 1))
print(f"  split at {state.index[half].date()}  LR={LR_h:.2f}  df~{df_h}  p={p_h:.4f}")
print(f"  cells pooled<5: {(Npool<5).mean():.0%} => chi2 {'UNRELIABLE' if (Npool<5).mean()>0.2 else 'ok'}")
print("  note: 2016-start sample; only COVID-2020 as crisis. limited power, suggestive only.")

[1] CIRCULAR-LAW DISK BENCHMARK
  avg transitions per origin row: 46.2
  noise disk radius ~ 1/sqrt(n_bar): 0.1472
  |lambda_2| observed: 0.0966
  verdict: INSIDE -> noise
  non-Perron moduli: 0.097, 0.097, 0.077, 0.077, 0.077, 0.051, 0.051, 0.047, 0.047, 0.009
  all non-Perron inside disk: True

[2] IRREDUCIBILITY & APERIODICITY
  raw P irreducible   : True
  smoothed P irreducible: True
  self-loops P[i,i]>0 (raw/smooth): 10 / 11 of 11
  period of chain (smoothed): 1 => aperiodic
  => irreducible + aperiodic = ergodic (unique pi, guaranteed convergence)

[3] MARKOV ORDER: order-0 (i.i.d.) vs order-1
  LR=87.68  df=100  p=0.8057  => order-0 sufficient (i.i.d.)
  AIC order0=2303.0  order1=2415.3  => order0
  BIC order0=2345.3  order1=2880.6  => order0

[4] ORDER-2 FEASIBILITY (sparsity wall)
  order-2 params: 121 pairs x 11 = 1331
  transitions available: 507 | distinct pairs seen: 114/121
  avg obs per pair: 4.45  => order-2 not estimable

[5] TIME-HOMOGENEITY (first-half vs second-ha

## Conclusion

Weekly laggard rotation across the 11 GICS sectors is statistically indistinguishable from i.i.d. Six independent tests agree: spectral gap, permutation, χ², daily re-test, block bootstrap, and raw-return lead-lag. The leader-state result that initially flagged under a simple shuffle collapses under a daily re-test and a block bootstrap.

This is a negative result for one specific framing: single worst-sector identity, weekly, first-order. It does not rule out sector rotation at longer horizons (6–12 month momentum) or under return-magnitude / regime-conditional definitions, where the literature locates what predictability exists.